Imports

In [ ]:
import os
import cv2
import numpy as np
import scipy.io
import random
import matplotlib.pyplot as plt
import mediapipe as mp
import shutil
from IPython.display import display, clear_output
import ipywidgets as widgets

In [ ]:
data_dir = '/content/drive/MyDrive/AFLW'

image_list = []
angles_list = []
filenames = []

for fname in sorted(os.listdir(data_dir)):
    if fname.endswith('.jpg'):
        img_path = os.path.join(data_dir, fname)
        mat_path = os.path.join(data_dir, fname.replace('.jpg', '.mat'))

        # Load image
        img = cv2.imread(img_path)
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        image_list.append(img_rgb)

        # Load pose
        label = scipy.io.loadmat(mat_path)
        pose = label['Pose_Para'].flatten()
        pitch, yaw, roll = np.degrees(pose[0]), np.degrees(pose[1]), np.degrees(pose[2])
        angles_list.append([pitch, yaw, roll])

        # Extract file number like '00001' from 'image00001.jpg'
        file_id = fname.replace('image', '').replace('.jpg', '')
        filenames.append(file_id)

# Convert angles to NumPy array
angles_np = np.array(angles_list)

# Confirm
print(f"Loaded {len(image_list)} images")
print(f"Angles shape: {angles_np.shape}")
print(f"Example filename: {filenames[0]}")


Loaded 2000 images
Angles shape: (2000, 3)
Example filename: 00002


In [ ]:
# Number of samples to show
num_samples = 10

for i in range(num_samples):
    img = image_list[i]
    angles = angles_np[i]
    fname = filenames[i]

    pitch, yaw, roll = angles

    plt.figure(figsize=(3, 3))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"{fname}\nPitch: {pitch:.1f}°, Yaw: {yaw:.1f}°, Roll: {roll:.1f}°")
    plt.show()


In [ ]:
pip install mediapipe opencv-python


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 28.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Uninstalling protobuf-5.29.4:
      Successfully uninstalled protobuf-5.29.4
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
grpcio-status 1.71.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.7 which is incompatib

Landmarks Visulisation

In [ ]:
# Initialize MediaPipe Face Mesh and process images
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
face_mesh = mp_face_mesh.FaceMesh(min_detection_confidence=0.8)

num_samples = 100
random_indices = random.sample(range(len(image_list)), num_samples)

for i in random_indices:
    img = image_list[i].copy()
    angles = angles_np[i]
    fname = filenames[i]

    # Convert the image to RGB (needed by MediaPipe)
    image_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Process the image for face landmarks
    results = face_mesh.process(image_rgb)

    feature_vector = []

    # Check if any landmarks were detected
    if results.multi_face_landmarks:
        # Process the first detected face
        face_landmarks = results.multi_face_landmarks[0]  # Only use the first face

        left_eye_pts = []
        right_eye_pts = []

        # Populate the left eye points list using a traditional for loop
        for i in [33, 133, 160, 144, 153, 158]:
            left_eye_pts.append((face_landmarks.landmark[i].x,
                                face_landmarks.landmark[i].y,
                                face_landmarks.landmark[i].z))

        # Populate the right eye points list using a traditional for loop
        for i in [362, 263, 387, 373, 380, 385]:
            right_eye_pts.append((face_landmarks.landmark[i].x,
                                  face_landmarks.landmark[i].y,
                                  face_landmarks.landmark[i].z))

        # Convert the lists into numpy arrays
        left_eye_center = np.mean(left_eye_pts, axis=0)
        right_eye_center = np.mean(right_eye_pts, axis=0)

        # Extract additional facial landmarks
        forehead = face_landmarks.landmark[10]  # Forehead center
        nose_tip = face_landmarks.landmark[1]   # Nose tip
        mouth_left = face_landmarks.landmark[61]  # Mouth left corner
        mouth_right = face_landmarks.landmark[291] # Mouth right corner
        chin = face_landmarks.landmark[152]      # Chin tip

        # Add new landmarks for left temporal and right temporal
        left_temporal = face_landmarks.landmark[137]
        right_temporal = face_landmarks.landmark[366]

        # Creating the feature list
        features = [
            ('Left Eye Center', left_eye_center[0], left_eye_center[1], left_eye_center[2]),
            ('Right Eye Center', right_eye_center[0], right_eye_center[1], right_eye_center[2]),
            ('Forehead Center', forehead.x, forehead.y, forehead.z),
            ('Nose Tip', nose_tip.x, nose_tip.y, nose_tip.z),
            ('Mouth Left Corner', mouth_left.x, mouth_left.y, mouth_left.z),
            ('Mouth Right Corner', mouth_right.x, mouth_right.y, mouth_right.z),
            ('Chin Tip', chin.x, chin.y, chin.z),
            ('Left Temporal', left_temporal.x, left_temporal.y, left_temporal.z),
            ('Right Temporal', right_temporal.x, right_temporal.y, right_temporal.z)
        ]

        # Add these features to the feature vector
        feature_vector.extend(features)

        for name, x, y, z in features:
            px, py = int(x * img.shape[1]), int(y * img.shape[0])  # Convert to pixel coordinates
            cv2.circle(img, (px, py), 3, (0, 255, 0), -1)

    # Display the image with landmarks
    image_rgb_with_landmarks = img
    plt.figure(figsize=(5, 5))
    plt.imshow(image_rgb_with_landmarks)
    plt.axis('off')
    plt.title(f"{fname}\nPitch: {angles[0]:.1f}°, Yaw: {angles[1]:.1f}°, Roll: {angles[2]:.1f}°")
    plt.show()

    # Print the feature vector (landmark names with coordinates)
    print(f"Selected Facial Landmarks for {fname}:")
    for name, x, y, z in feature_vector:
        print(f"{name}: x={x:.3f}, y={y:.3f}, z={z:.3f}")
    print("\n---\n")

In [ ]:
# Directory where your images are stored
data_dir = '/content/drive/MyDrive/AFLW'

# Initialize MediaPipe Face Mesh model
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils

# Initialize face mesh
face_mesh = mp_face_mesh.FaceMesh(min_detection_confidence=0.8)

num_samples = 30

# Randomly select images and their corresponding angles
random_indices = random.sample(range(len(image_list)), num_samples)

for i in random_indices:
    img2 = image_list[i].copy()
    angles = angles_np[i]
    fname = filenames[i]

    img_path = os.path.join(data_dir, f"image{fname}.jpg")

    # Load the image
    img = cv2.imread(img_path)
    if img is None:
        print(f"Failed to load {img_path}")
        continue

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Process the image to detect landmarks
    results = face_mesh.process(img_rgb)

    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            # Draw all landmarks on the image (for visualization)
            mp_drawing.draw_landmarks(img, face_landmarks, mp_face_mesh.FACEMESH_CONTOURS)

            # Print coordinates of all landmarks


        # Convert image to RGB for display with matplotlib
        img_rgb_with_landmarks = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Show the image with landmarks
        plt.figure(figsize=(5, 5))
        plt.imshow(img_rgb_with_landmarks)
        plt.axis('off')
        plt.title(f"Landmarks for {fname}")
        plt.show()

    else:
        print(f"No landmarks found for {fname}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Load Trained model File For Dlib to work

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d sergiovirahonda/shape-predictor-68-face-landmarksdat




Dataset URL: https://www.kaggle.com/datasets/sergiovirahonda/shape-predictor-68-face-landmarksdat
License(s): unknown


Manual Feature Selection

In [ ]:
# Initialize MediaPipe Face Mesh model
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(min_detection_confidence=0.8, min_tracking_confidence=0.5)

# Paths
data_dir = '/content/drive/MyDrive/AFLW'
"""
valid_dir = os.path.join(data_dir, 'valid')
corrupt_dir = os.path.join(data_dir, 'corrupt')
"""
valid_dir = '/content/AFLW2000/valid'
corrupt_dir = '/content/AFLW2000/corrupt'
os.makedirs(valid_dir, exist_ok=True)
os.makedirs(corrupt_dir, exist_ok=True)

# Get image list
image_files = [f for f in os.listdir(data_dir) if f.endswith('.jpg')]
image_index = 0  # Global index
quit_flag = False  # Exit condition

def process_image(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None, None

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(img_rgb)
    feature_vector = []

    if results.multi_face_landmarks:
        # Process only the first detected face
        face_landmarks = results.multi_face_landmarks[0]

        # Collect eye landmarks using traditional for loops
        left_eye_pts = []
        for i in [33, 133, 160, 144, 153, 158]:
            left_eye_pts.append((face_landmarks.landmark[i].x,
                                 face_landmarks.landmark[i].y,
                                 face_landmarks.landmark[i].z))

        right_eye_pts = []
        for i in [362, 263, 387, 373, 380, 385]:
            right_eye_pts.append((face_landmarks.landmark[i].x,
                                  face_landmarks.landmark[i].y,
                                  face_landmarks.landmark[i].z))

        left_eye_center = np.mean(left_eye_pts, axis=0)
        right_eye_center = np.mean(right_eye_pts, axis=0)

        # Extract other landmarks
        forehead = face_landmarks.landmark[10]
        nose_tip = face_landmarks.landmark[1]
        mouth_left = face_landmarks.landmark[61]
        mouth_right = face_landmarks.landmark[291]
        chin = face_landmarks.landmark[152]
        left_temporal = face_landmarks.landmark[137]
        right_temporal = face_landmarks.landmark[366]

        # Build the feature vector
        features = [
            ('Left Eye Center', left_eye_center[0], left_eye_center[1], left_eye_center[2]),
            ('Right Eye Center', right_eye_center[0], right_eye_center[1], right_eye_center[2]),
            ('Forehead Center', forehead.x, forehead.y, forehead.z),
            ('Nose Tip', nose_tip.x, nose_tip.y, nose_tip.z),
            ('Mouth Left Corner', mouth_left.x, mouth_left.y, mouth_left.z),
            ('Mouth Right Corner', mouth_right.x, mouth_right.y, mouth_right.z),
            ('Chin Tip', chin.x, chin.y, chin.z),
            ('Left Temporal', left_temporal.x, left_temporal.y, left_temporal.z),
            ('Right Temporal', right_temporal.x, right_temporal.y, right_temporal.z)
        ]

        feature_vector.extend(features)

        # Draw the keypoints on the image
        for label, x, y, z in features:
            px = int(x * img.shape[1])
            py = int(y * img.shape[0])
            cv2.circle(img, (px, py), 3, (0, 255, 0), -1)

    return img, feature_vector


def display_image(img, fname, feature_vector):
    plt.figure(figsize=(10, 8))
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f"Image: {fname}")

    plt.subplot(1, 2, 2)
    plt.axis('off')
    text = "Facial Landmarks:\n"
    for name, x, y, z in feature_vector:
        text += f"{name}:\n x={x:.3f}, y={y:.3f}, z={z:.3f}\n\n"
    plt.text(0, 0.5, text, fontsize=9, va='center')

    plt.tight_layout()
    plt.show()

def handle_decision(decision):
    global image_index, quit_flag

    fname = image_files[image_index]
    src_path = os.path.join(data_dir, fname)

    if decision == 'valid':
        shutil.copy2(src_path, os.path.join(valid_dir, fname))
        mat_file = fname.replace('.jpg', '.mat')
        if os.path.exists(os.path.join(data_dir, mat_file)):
            shutil.copy2(os.path.join(data_dir, mat_file), os.path.join(valid_dir, mat_file))

    elif decision == 'corrupt':
        shutil.copy2(src_path, os.path.join(corrupt_dir, fname))
        mat_file = fname.replace('.jpg', '.mat')
        if os.path.exists(os.path.join(data_dir, mat_file)):
            shutil.copy2(os.path.join(data_dir, mat_file), os.path.join(corrupt_dir, mat_file))

    elif decision == 'quit':
        quit_flag = True

    image_index += 1
    show_next_image()

def show_next_image():
    clear_output(wait=True)
    global image_index

    if quit_flag or image_index >= len(image_files):
        print("✅ Annotation completed or quit by user.")
        return

    fname = image_files[image_index]
    img_path = os.path.join(data_dir, fname)
    print(f"\nProcessing image {image_index + 1}/{len(image_files)}: {fname}")
    img, feature_vector = process_image(img_path)

    if img is None:
        print("Image is unreadable, skipping...")
        image_index += 1
        show_next_image()
        return

    display_image(img, fname, feature_vector)
    display(widgets.HBox([valid_button, corrupt_button, quit_button]))

# Buttons
valid_button = widgets.Button(description='✅ Valid', button_style='success')
corrupt_button = widgets.Button(description='❌ Corrupt', button_style='danger')
quit_button = widgets.Button(description='⏹️ Quit', button_style='warning')

valid_button.on_click(lambda b: handle_decision('valid'))
corrupt_button.on_click(lambda b: handle_decision('corrupt'))
quit_button.on_click(lambda b: handle_decision('quit'))

# Start processing
show_next_image()


In [ ]:
drive_target_dir = '/content/drive/MyDrive/FaceAnnotations2'

# Create target directories if they don't exist
os.makedirs(os.path.join(drive_target_dir, 'valid'), exist_ok=True)
os.makedirs(os.path.join(drive_target_dir, 'corrupt'), exist_ok=True)

# Copy folders
shutil.copytree('/content/AFLW2000/valid', os.path.join(drive_target_dir, 'valid'), dirs_exist_ok=True)
shutil.copytree('/content/AFLW2000/corrupt', os.path.join(drive_target_dir, 'corrupt'), dirs_exist_ok=True)

print("✅ Folders successfully copied to Google Drive!")


✅ Folders successfully copied to Google Drive!
